# 🏗️ ArchAI — Blender Server (Google Colab)

**AI-архитектор: генерация 3D-зданий + фотореалистичный интерьерный дизайн**

Этот ноутбук запускает Blender headless-сервер, который:
1. Принимает параметры здания из веб-интерфейса ArchAI
2. Генерирует 3D-модель через Blender Python API
3. Рендерит фотореалистичные интерьеры
4. Экспортирует в GLB/FBX/OBJ/Blend

---

**Интегрировано с:**
- [BlenderLLM](https://github.com/FreedomIntelligence/BlenderLLM) — LLM для CAD-скриптов
- [StableGen](https://github.com/sakalond/StableGen) — AI-текстурирование
- [BlenderProc](https://github.com/DLR-RM/BlenderProc) — фотореалистичный рендеринг

## 📦 Шаг 1: Установка окружения

In [ ]:
#@title Установка Blender и зависимостей { display-mode: "form" }
!apt-get update -qq && apt-get install -y -qq blender > /dev/null 2>&1
!pip install -q flask flask-cors pyngrok

# Проверяем Blender
!blender --version
print("\n✅ Blender установлен!")

In [ ]:
#@title Загрузка скриптов ArchAI { display-mode: "form" }
import os, json

# Создаём директорию
!mkdir -p /content/archai_blender/output

# Скачиваем скрипты из репозитория
# (или копируем напрямую если репо уже клонирован)
REPO_URL = "https://raw.githubusercontent.com/smartmoneymoscow-cell/AI_Arhitector/main/blender"

!curl -sL "{REPO_URL}/generate_building.py" -o /content/archai_blender/generate_building.py
!curl -sL "{REPO_URL}/render_interior.py" -o /content/archai_blender/render_interior.py

print("✅ Скрипты загружены:")
!ls -la /content/archai_blender/*.py

## 🏠 Шаг 2: Генерация здания

In [ ]:
#@title Параметры здания { display-mode: "form" }

building_type = "house"  #@param ["house", "cottage", "office", "villa"]
floors = 2  #@param {type:"slider", min:1, max:10, step:1}
width = 10  #@param {type:"slider", min:4, max:30, step:1}
length = 12  #@param {type:"slider", min:4, max:30, step:1}
roof_type = "gabled"  #@param ["gabled", "hip", "flat"]
facade_material = "brick"  #@param ["brick", "wood", "glass", "plaster", "stone", "concrete"]
facade_color = "#e8e0d4"  #@param {type:"string"}
has_balcony = True  #@param {type:"boolean"}
has_terrace = False  #@param {type:"boolean"}
has_garage = True  #@param {type:"boolean"}
style = "modern"  #@param ["modern", "classic", "scandinavian", "hitech", "barnhouse", "chalet"]

params = {
    "type": building_type,
    "floors": floors,
    "width": width,
    "length": length,
    "roof_type": roof_type,
    "facade_material": facade_material,
    "facade_color": facade_color,
    "has_balcony": has_balcony,
    "has_terrace": has_terrace,
    "has_garage": has_garage,
    "style": style,
}

with open('/content/archai_blender/params.json', 'w') as f:
    json.dump(params, f, indent=2, ensure_ascii=False)

print("📋 Параметры здания:")
print(json.dumps(params, indent=2, ensure_ascii=False))

In [ ]:
#@title Генерация 3D-модели здания { display-mode: "form" }
export_format = "glb"  #@param ["glb", "fbx", "obj", "blend"]

output_file = f"/content/archai_blender/output/building.{export_format}"

!blender --background --factory-startup --python /content/archai_blender/generate_building.py -- /content/archai_blender/params.json "{output_file}"

print(f"\n✅ Модель экспортирована: {output_file}")
print(f"📁 Размер: {os.path.getsize(output_file) / 1024:.1f} KB")

In [ ]:
#@title Предпросмотр (рендер камеры) { display-mode: "form" }

!blender --background --factory-startup /content/archai_blender/output/building.blend \
    --python-expr "
import bpy
bpy.context.scene.render.engine = 'CYCLES'
bpy.context.scene.cycles.samples = 64
bpy.context.scene.render.resolution_x = 1280
bpy.context.scene.render.resolution_y = 720
bpy.context.scene.render.filepath = '/content/archai_blender/output/preview.png'
bpy.ops.render.render(write_still=True)
" 2>/dev/null

# Показываем результат
from IPython.display import Image, display
if os.path.exists('/content/archai_blender/output/preview.png'):
    display(Image('/content/archai_blender/output/preview.png', width=800))
else:
    print("⚠️ Рендер не удался. Попробуйте формат .blend и откройте его локально.")

## 🛋️ Шаг 3: Фотореалистичный интерьер

In [ ]:
#@title Параметры интерьера { display-mode: "form" }

room_type = "living_room"  #@param ["living_room", "bedroom", "kitchen", "office", "bathroom"]
room_width = 6  #@param {type:"slider", min:3, max:15, step:1}
room_length = 8  #@param {type:"slider", min:3, max:15, step:1}
room_height = 3  #@param {type:"slider", min:2.5, max:5, step:0.5}
interior_style = "modern"  #@param ["modern", "classic", "scandinavian", "loft", "minimalist"]
camera_view = "corner"  #@param ["corner", "center"]
render_samples = 256  #@param ["64", "128", "256", "512"]
render_resolution = "1920x1080"  #@param ["1280x720", "1920x1080", "2560x1440"]

res_w, res_h = map(int, render_resolution.split('x'))

furniture_map = {
    "living_room": ["sofa", "table", "chandelier"],
    "bedroom": ["bed", "chandelier"],
    "kitchen": ["table", "chandelier"],
    "office": ["table", "chandelier"],
    "bathroom": [],
}

interior_params = {
    "room_type": room_type,
    "width": room_width,
    "length": room_length,
    "height": room_height,
    "style": interior_style,
    "furniture": furniture_map.get(room_type, []),
    "camera_position": camera_view,
    "samples": int(render_samples),
    "resolution": res_w,
}

with open('/content/archai_blender/interior_params.json', 'w') as f:
    json.dump(interior_params, f, indent=2, ensure_ascii=False)

print("📋 Параметры интерьера:")
print(json.dumps(interior_params, indent=2, ensure_ascii=False))

In [ ]:
#@title Рендер интерьера { display-mode: "form" }

interior_output = "/content/archai_blender/output/interior.png"

!blender --background --factory-startup --python /content/archai_blender/render_interior.py -- /content/archai_blender/interior_params.json "{interior_output}"

# Показываем результат
from IPython.display import Image, display
if os.path.exists(interior_output):
    print(f"\n✅ Интерьер отрендерен: {interior_output}")
    display(Image(interior_output, width=900))
else:
    print("⚠️ Рендер не удался. Проверьте логи выше.")

## 🌐 Шаг 4: API-сервер (для веб-интерфейса)

In [ ]:
#@title Запуск Flask API сервера { display-mode: "form" }

from flask import Flask, request, jsonify, send_file
from flask_cors import CORS
from pyngrok import ngrok
import subprocess, uuid, threading

app = Flask(__name__)
CORS(app)

OUTPUT_DIR = "/content/archai_blender/output"
BLENDER_SCRIPTS = "/content/archai_blender"

@app.route('/health')
def health():
    return jsonify({"status": "ok", "service": "archai-blender"})

@app.route('/api/v1/generate/building', methods=['POST'])
def generate_building():
    params = request.json
    job_id = str(uuid.uuid4())[:8]
    fmt = params.pop('export_format', 'glb')
    output_file = f"{OUTPUT_DIR}/{job_id}.{fmt}"
    params_file = f"{OUTPUT_DIR}/{job_id}.json"

    with open(params_file, 'w') as f:
        json.dump(params, f)

    result = subprocess.run(
        ['blender', '--background', '--factory-startup',
         '--python', f'{BLENDER_SCRIPTS}/generate_building.py',
         '--', params_file, output_file],
        capture_output=True, text=True, timeout=120
    )

    if os.path.exists(output_file):
        return send_file(output_file, as_attachment=True,
                        download_name=f"archai_building.{fmt}")
    return jsonify({"error": result.stderr[-500:]}), 500

@app.route('/api/v1/render/interior', methods=['POST'])
def render_interior():
    params = request.json
    job_id = str(uuid.uuid4())[:8]
    output_file = f"{OUTPUT_DIR}/{job_id}_interior.png"
    params_file = f"{OUTPUT_DIR}/{job_id}_interior.json"

    with open(params_file, 'w') as f:
        json.dump(params, f)

    result = subprocess.run(
        ['blender', '--background', '--factory-startup',
         '--python', f'{BLENDER_SCRIPTS}/render_interior.py',
         '--', params_file, output_file],
        capture_output=True, text=True, timeout=300
    )

    if os.path.exists(output_file):
        return send_file(output_file, as_attachment=True,
                        download_name=f"archai_interior.png")
    return jsonify({"error": result.stderr[-500:]}), 500

@app.route('/api/v1/export/<fmt>', methods=['POST'])
def export_model(fmt):
    """Convert between formats using Blender."""
    if request.content_type and 'multipart' in request.content_type:
        file = request.files.get('model')
        if file:
            input_path = f"{OUTPUT_DIR}/upload_{uuid.uuid4().hex[:8]}.blend"
            file.save(input_path)
            output_path = input_path.replace('.blend', f'.{fmt}')
            subprocess.run(
                ['blender', '--background', '--factory-startup', input_path,
                 '--python-expr', f"import bpy; bpy.ops.export_scene.gltf(filepath='{output_path}')"],
                capture_output=True, timeout=60
            )
            if os.path.exists(output_path):
                return send_file(output_path, as_attachment=True)
    return jsonify({"error": "No model provided"}), 400

# Запускаем ngrok туннель
public_url = ngrok.connect(5000).public_url
print(f"\n{'='*60}")
print(f"🌐 ArchAI Blender Server запущен!")
print(f"📡 Public URL: {public_url}")
print(f"{'='*60}")
print(f"\n📋 Endpoints:")
print(f"  GET  {public_url}/health")
print(f"  POST {public_url}/api/v1/generate/building")
print(f"  POST {public_url}/api/v1/render/interior")
print(f"  POST {public_url}/api/v1/export/<format>")
print(f"\n💡 Скопируйте URL и вставьте в веб-интерфейс ArchAI")

app.run(port=5000, host='0.0.0.0')

## 🧪 Шаг 5: Быстрый тест

In [ ]:
#@title Тест генерации здания из текста { display-mode: "form" }

user_prompt = "Двухэтажный кирпичный дом 10×12 с двускатной кровлей и балконом"  #@param {type:"string"}

# Парсим параметры из текста (локальный парсер)
import re

def parse_building_params(text):
    t = text.lower()
    p = {}
    
    # Этажность
    floors_m = re.search(r'(\d+)\s*(?:этаж|floor)', t)
    if floors_m: p['floors'] = int(floors_m.group(1))
    for word, num in [('двух',2),('трех',3),('четыр',4),('пяти',5)]:
        if word in t and 'этаж' in t: p['floors'] = num
    
    # Размеры
    dim_m = re.search(r'(\d+)\s*[×xх]\s*(\d+)', t)
    if dim_m:
        p['width'] = int(dim_m.group(1))
        p['length'] = int(dim_m.group(2))
    
    # Материал
    for mat in ['кирпич', 'дерев', 'стекл', 'камен', 'бетон']:
        if mat in t:
            p['facade_material'] = {'кирпич':'brick','дерев':'wood','стекл':'glass','камен':'stone','бетон':'concrete'}[mat]
    
    # Кровля
    if 'двускатн' in t or 'скатн' in t: p['roof_type'] = 'gabled'
    elif 'плоск' in t: p['roof_type'] = 'flat'
    elif 'вальм' in t: p['roof_type'] = 'hip'
    
    # Опции
    p['has_balcony'] = 'балкон' in t
    p['has_terrace'] = 'террас' in t
    p['has_garage'] = 'гараж' in t
    
    return p

parsed = parse_building_params(user_prompt)
# Merge with defaults
full_params = {
    "type": "house", "floors": 2, "width": 10, "length": 12,
    "roof_type": "gabled", "facade_material": "brick",
    "facade_color": "#e8e0d4", "has_balcony": False,
    "has_terrace": False, "has_garage": False,
}
full_params.update(parsed)

print(f"📝 Промт: {user_prompt}")
print(f"📋 Распознанные параметры: {json.dumps(parsed, ensure_ascii=False)}")
print(f"\n🚀 Генерация...")

test_output = "/content/archai_blender/output/test_building.glb"
with open('/content/archai_blender/test_params.json', 'w') as f:
    json.dump(full_params, f)

!blender --background --factory-startup --python /content/archai_blender/generate_building.py -- /content/archai_blender/test_params.json "{test_output}" 2>&1 | tail -5

if os.path.exists(test_output):
    print(f"\n✅ Готово! Файл: {test_output} ({os.path.getsize(test_output)/1024:.1f} KB)")
    print(f"📥 Скачайте файл из панели файлов слева → archai_blender/output/")
else:
    print("❌ Ошибка генерации")

## 📝 Как использовать с веб-интерфейсом

1. Запустите **Шаг 4** (API сервер) — получите `Public URL`
2. Откройте веб-интерфейс ArchAI
3. В настройках укажите URL сервера (из шага 4)
4. Опишите здание или интерьер — сервер сгенерирует модель в Blender

### Текстовые промты для зданий:
- `двухэтажный кирпичный дом 10×12 с двускатной кровлей`
- `современный офис 5 этажей стекло 15×20`
- `деревянный коттедж с террасой и гаражом`

### Текстовые промты для интерьеров:
- `гостиная в скандинавском стиле 6×8`
- `спальня в стиле лофт 4×5`
- `минималистичная кухня 3×4`